In [1]:
import json
import os
from pathlib import Path

import pandas as pd

In [2]:
RESULTS_DIR = Path(os.path.expanduser("~/scFM_eval/results"))
TABLE_DIR = Path("./tables")
TABLE_DIR.mkdir(parents=True, exist_ok=True)

CLASSIFIER = "randomforest"
EXCLUDE_EXPS = {"luad1", "luad_cancer_stage"}
TABLE_COLUMNS = ["experiment_name", "experiment", "group", "mean", "task"]


def load_classification_metrics(results_dir: Path) -> pd.DataFrame:
    """Load fold-level classification metrics from CSV or JSON exports."""
    csv_path = results_dir / "classification.metrics.csv"
    json_path = results_dir / "classification.metrics.json"
    if csv_path.exists():
        return pd.read_csv(csv_path)
    if json_path.exists():
        payload = json.loads(json_path.read_text())
        return pd.DataFrame(payload["records"])
    raise FileNotFoundError(
        f"No classification.metrics.csv or .json in {results_dir}"
    )


def format_auprc_table(metrics: pd.DataFrame, strategy: str) -> pd.DataFrame:
    """Mean AUPRC across CV folds for one aggregation strategy."""
    sub = metrics[metrics["strategy"] == strategy].copy()
    out = (
        sub.groupby(
            ["exp", "exp_display", "model_display", "group"],
            as_index=False,
        )
        .agg(mean=("AUPRC", "mean"))
        .rename(
            columns={
                "exp": "experiment_name",
                "model_display": "experiment",
                "exp_display": "task",
            }
        )
    )
    out = out.round(3)
    return out[TABLE_COLUMNS].sort_values(
        ["experiment_name", "group", "experiment"]
    ).reset_index(drop=True)


raw = load_classification_metrics(RESULTS_DIR)
metrics = raw[raw["classifier"] == CLASSIFIER].copy()
metrics = metrics[~metrics["exp"].isin(EXCLUDE_EXPS)].copy()
metrics = metrics.dropna(subset=["model_display", "exp_display", "group"])

print(
    f"{metrics['model_display'].nunique()} models, "
    f"{metrics['exp_display'].nunique()} tasks, "
    f"{len(metrics)} fold rows from {RESULTS_DIR}"
)
print("strategies:", sorted(metrics["strategy"].unique()))

17 models, 7 tasks, 1734 fold rows from /home/haitham/scFM_eval/results
strategies: ['MIL', 'avg', 'vote']


In [3]:
df = format_auprc_table(metrics, "avg")
print(df)
df.to_csv(TABLE_DIR / "Table5_AUPRC_avg.csv", index=False)

       experiment_name      experiment     group   mean  \
0           brca_chemo             HVG  Baseline  0.633   
1           brca_chemo       PCA [100]  Baseline  0.492   
2           brca_chemo        PCA [20]  Baseline  0.398   
3           brca_chemo        PCA [50]  Baseline  0.647   
4           brca_chemo            scVI  Baseline  0.532   
..                 ...             ...       ...    ...   
114  melanoma_response           STATE     Other  0.950   
115  melanoma_response       scConcept     Other  0.917   
116  melanoma_response    scFoundation     Other  0.867   
117  melanoma_response           scGPT     scGPT  0.817   
118  melanoma_response  scGPT [cancer]     scGPT  0.917   

                                     task  
0    Treatment Naive vs Neoadjuvant Chemo  
1    Treatment Naive vs Neoadjuvant Chemo  
2    Treatment Naive vs Neoadjuvant Chemo  
3    Treatment Naive vs Neoadjuvant Chemo  
4    Treatment Naive vs Neoadjuvant Chemo  
..                         

In [4]:
df = format_auprc_table(metrics, "vote")
print(df)
df.to_csv(TABLE_DIR / "Table6_AUPRC_vote.csv", index=False)

       experiment_name      experiment     group   mean  \
0           brca_chemo             HVG  Baseline  0.548   
1           brca_chemo       PCA [100]  Baseline  0.490   
2           brca_chemo        PCA [20]  Baseline  0.438   
3           brca_chemo        PCA [50]  Baseline  0.377   
4           brca_chemo            scVI  Baseline  0.632   
..                 ...             ...       ...    ...   
114  melanoma_response           STATE     Other  0.950   
115  melanoma_response       scConcept     Other  0.950   
116  melanoma_response    scFoundation     Other  0.917   
117  melanoma_response           scGPT     scGPT  0.950   
118  melanoma_response  scGPT [cancer]     scGPT  0.950   

                                     task  
0    Treatment Naive vs Neoadjuvant Chemo  
1    Treatment Naive vs Neoadjuvant Chemo  
2    Treatment Naive vs Neoadjuvant Chemo  
3    Treatment Naive vs Neoadjuvant Chemo  
4    Treatment Naive vs Neoadjuvant Chemo  
..                         

In [5]:
df = format_auprc_table(metrics, "MIL")
print(df)
df.to_csv(TABLE_DIR / "Table7_AUPRC_mil.csv", index=False)

       experiment_name      experiment     group   mean  \
0           brca_chemo             HVG  Baseline  0.587   
1           brca_chemo       PCA [100]  Baseline  0.773   
2           brca_chemo        PCA [20]  Baseline  0.729   
3           brca_chemo        PCA [50]  Baseline  0.633   
4           brca_chemo            scVI  Baseline  0.673   
..                 ...             ...       ...    ...   
114  melanoma_response           STATE     Other  0.867   
115  melanoma_response       scConcept     Other  0.950   
116  melanoma_response    scFoundation     Other  0.950   
117  melanoma_response           scGPT     scGPT  0.883   
118  melanoma_response  scGPT [cancer]     scGPT  0.917   

                                     task  
0    Treatment Naive vs Neoadjuvant Chemo  
1    Treatment Naive vs Neoadjuvant Chemo  
2    Treatment Naive vs Neoadjuvant Chemo  
3    Treatment Naive vs Neoadjuvant Chemo  
4    Treatment Naive vs Neoadjuvant Chemo  
..                         

In [6]:
df

,experiment_name,experiment,group,mean,task
0,brca_chemo,HVG,Baseline,0.587,Treatment Naive vs Neoadjuvant Chemo
1,brca_chemo,PCA [100],Baseline,0.773,Treatment Naive vs Neoadjuvant Chemo
2,brca_chemo,PCA [20],Baseline,0.729,Treatment Naive vs Neoadjuvant Chemo
3,brca_chemo,PCA [50],Baseline,0.633,Treatment Naive vs Neoadjuvant Chemo
4,brca_chemo,scVI,Baseline,0.673,Treatment Naive vs Neoadjuvant Chemo
...,...,...,...,...,...
114,melanoma_response,STATE,Other,0.867,IO Response
115,melanoma_response,scConcept,Other,0.950,IO Response
116,melanoma_response,scFoundation,Other,0.950,IO Response
117,melanoma_response,scGPT,scGPT,0.883,IO Response
